# **Labs 1 PySpark:**

In these labs we will be using the "[[NeurIPS 2020] Data Science for COVID-19 (DS4C)](https://www.kaggle.com/datasets/kimjihoo/coronavirusdataset?select=PatientInfo.csv)" dataset, retrieved from [Kaggle](https://www.kaggle.com/) on 1/6/2022, for educational non commercial purpose, License
[CC BY-NC-SA 4.0
](https://creativecommons.org/licenses/by-nc-sa/4.0/)


The csv file that we will be using in this lab is **PatientInfo**.

## PatientInfo.csv

**patient_id**
the ID of the patient

**sex**
the sex of the patient

**age**
the age of the patient

**country**
the country of the patient

**province**
the province of the patient

**city**
the city of the patient

**infection_case**
the case of infection

**infected_by**
the ID of who infected the patient


**contact_number**
the number of contacts with people

**symptom_onset_date**
the date of symptom onset

**confirmed_date**
the date of being confirmed

**released_date**
the date of being released

**deceased_date**
the date of being deceased

**state**
isolated / released / deceased

### Import the pyspark and check it's version

In [1]:
import pyspark
print(pyspark.__version__)

4.0.2


### Import and create SparkSession

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Covid19_Analysis').getOrCreate()

### Load the PatientInfo.csv file and show the first 5 rows

In [ ]:
from IPython.display import display, HTML
display(HTML("<style>pre { white-space: pre !important; }</style>"))

In [ ]:
df = spark.read.csv('/content/PatientInfo.csv', header=True, inferSchema=True)
df.show(5)

+----------+------+---+-------+--------+-----------+--------------------+-----------+--------------+------------------+--------------+-------------+-------------+--------+
|patient_id|   sex|age|country|province|       city|      infection_case|infected_by|contact_number|symptom_onset_date|confirmed_date|released_date|deceased_date|   state|
+----------+------+---+-------+--------+-----------+--------------------+-----------+--------------+------------------+--------------+-------------+-------------+--------+
|1000000001|  male|50s|  Korea|   Seoul| Gangseo-gu|     overseas inflow|       NULL|            75|        2020-01-22|    2020-01-23|   2020-02-05|         NULL|released|
|1000000002|  male|30s|  Korea|   Seoul|Jungnang-gu|     overseas inflow|       NULL|            31|              NULL|    2020-01-30|   2020-03-02|         NULL|released|
|1000000003|  male|50s|  Korea|   Seoul|  Jongno-gu|contact with patient| 2002000001|            17|              NULL|    2020-01-30|   202

### Display the schema of the dataset

In [ ]:
df.printSchema()

root
 |-- patient_id: long (nullable = true)
 |-- sex: string (nullable = true)
 |-- age: string (nullable = true)
 |-- country: string (nullable = true)
 |-- province: string (nullable = true)
 |-- city: string (nullable = true)
 |-- infection_case: string (nullable = true)
 |-- infected_by: string (nullable = true)
 |-- contact_number: string (nullable = true)
 |-- symptom_onset_date: string (nullable = true)
 |-- confirmed_date: date (nullable = true)
 |-- released_date: date (nullable = true)
 |-- deceased_date: date (nullable = true)
 |-- state: string (nullable = true)



### Display the statistical summary

In [ ]:
df.describe().show()

+-------+--------------------+------+----+----------+--------+--------------+--------------------+--------------------+--------------------+------------------+--------+
|summary|          patient_id|   sex| age|   country|province|          city|      infection_case|         infected_by|      contact_number|symptom_onset_date|   state|
+-------+--------------------+------+----+----------+--------+--------------+--------------------+--------------------+--------------------+------------------+--------+
|  count|                5165|  4043|3785|      5165|    5165|          5071|                4246|                1346|                 791|               690|    5165|
|   mean|2.8636345618679576E9|  NULL|NULL|      NULL|    NULL|          NULL|                NULL|2.2845944015643125E9|1.6772572523506988E7|              NULL|    NULL|
| stddev| 2.074210725277473E9|  NULL|NULL|      NULL|    NULL|          NULL|                NULL|1.5265072953383324E9| 3.093097580985502E8|              N

### Using the state column.
### How many people survived (released), and how many didn't survive (isolated/deceased)?

In [ ]:
df.groupBy('state').count().show()

+--------+-----+
|   state|count|
+--------+-----+
|isolated| 2158|
|released| 2929|
|deceased|   78|
+--------+-----+



### Display the number of null values in each column

In [ ]:
from pyspark.sql.functions import col, count, when
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

+----------+----+----+-------+--------+----+--------------+-----------+--------------+------------------+--------------+-------------+-------------+-----+
|patient_id| sex| age|country|province|city|infection_case|infected_by|contact_number|symptom_onset_date|confirmed_date|released_date|deceased_date|state|
+----------+----+----+-------+--------+----+--------------+-----------+--------------+------------------+--------------+-------------+-------------+-----+
|         0|1122|1380|      0|       0|  94|           919|       3819|          4374|              4475|             3|         3578|         5099|    0|
+----------+----+----+-------+--------+----+--------------+-----------+--------------+------------------+--------------+-------------+-------------+-----+



## Data preprocessing

### Fill the nulls in the deceased_date with the released_date.
- You can use <b>coalesce</b> function

In [ ]:
from pyspark.sql.functions import coalesce

df2 = df.withColumn('deceased_date', coalesce(df['deceased_date'], df['released_date']))

### Add a column named no_days which is difference between the deceased_date and the confirmed_date then show the top 5 rows. Print the schema.
- <b> Hint: You need to typecast these columns as date first <b>

In [ ]:
from pyspark.sql.functions import datediff, to_date
df3 = df2.withColumn('no_days', datediff(to_date(df2['deceased_date']), to_date(df2['confirmed_date'])))
df3.show(5)
df3.printSchema()

+----------+------+---+-------+--------+-----------+--------------------+-----------+--------------+------------------+--------------+-------------+-------------+--------+-------+
|patient_id|   sex|age|country|province|       city|      infection_case|infected_by|contact_number|symptom_onset_date|confirmed_date|released_date|deceased_date|   state|no_days|
+----------+------+---+-------+--------+-----------+--------------------+-----------+--------------+------------------+--------------+-------------+-------------+--------+-------+
|1000000001|  male|50s|  Korea|   Seoul| Gangseo-gu|     overseas inflow|       NULL|            75|        2020-01-22|    2020-01-23|   2020-02-05|   2020-02-05|released|     13|
|1000000002|  male|30s|  Korea|   Seoul|Jungnang-gu|     overseas inflow|       NULL|            31|              NULL|    2020-01-30|   2020-03-02|   2020-03-02|released|     32|
|1000000003|  male|50s|  Korea|   Seoul|  Jongno-gu|contact with patient| 2002000001|            17|

### Remove null values of sex column.
### Add a is_male column if male then it should yield true, else (Female) then False

In [ ]:
df4 = df3.dropna(subset=['sex'])
df5 = df4.withColumn('is_male', when(df4['sex'] == 'male', True).otherwise(False))

### Add a is_dead column if patient state is not released then it should yield true, else then False

- Use <b>UDF</b> to perform this task.
- However, UDF is not recommended there is no built in function can do the required operation.
- UDF is slower than built in functions.

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import BooleanType

is_dead_udf = udf(lambda state: state != 'released', BooleanType())
df6 = df5.withColumn('is_dead', is_dead_udf(df['state']))

### Change the ages to bins from 10s, 0s, 10s, 20s,.etc to 0,10, 20

In [ ]:
from pyspark.sql.functions import regexp_replace

df7 = df6.withColumn('age', regexp_replace('age', 's', ''))

### Change age, and no_days  to be typecasted as Double

In [ ]:
df8 = df7.withColumn('age', df7['age'].cast('double')) \
       .withColumn('no_days', df7['no_days'].cast('double'))

### Drop the columns
["patient_id","sex","infected_by","contact_number","released_date","state",
"symptom_onset_date","confirmed_date","deceased_date","country","no_days",
"city","infection_case"]

In [ ]:
cols_to_drop = ["patient_id","sex","infected_by","contact_number","released_date","state", "symptom_onset_date","confirmed_date","deceased_date","country","no_days", "city","infection_case"]
df_cleaned = df8.drop(*cols_to_drop)

### Recount the number of nulls now

In [ ]:
df_cleaned.select([count(when(col(c).isNull(), c)).alias(c) for c in df_cleaned.columns]).show()

+---+--------+-------+-------+
|age|province|is_male|is_dead|
+---+--------+-------+-------+
|261|       0|      0|      0|
+---+--------+-------+-------+



## Now do the same but using SQL select statement

### From the original Patient DataFrame, Create a temporary view (table).

In [ ]:
df.createOrReplaceTempView('patients')

### Use SELECT statement to select all columns from the dataframe and show the output.

In [ ]:
spark.sql("SELECT * FROM patients").show()

+----------+------+---+-------+--------+------------+--------------------+-----------+--------------+------------------+--------------+-------------+-------------+--------+
|patient_id|   sex|age|country|province|        city|      infection_case|infected_by|contact_number|symptom_onset_date|confirmed_date|released_date|deceased_date|   state|
+----------+------+---+-------+--------+------------+--------------------+-----------+--------------+------------------+--------------+-------------+-------------+--------+
|1000000001|  male|50s|  Korea|   Seoul|  Gangseo-gu|     overseas inflow|       NULL|            75|        2020-01-22|    2020-01-23|   2020-02-05|         NULL|released|
|1000000002|  male|30s|  Korea|   Seoul| Jungnang-gu|     overseas inflow|       NULL|            31|              NULL|    2020-01-30|   2020-03-02|         NULL|released|
|1000000003|  male|50s|  Korea|   Seoul|   Jongno-gu|contact with patient| 2002000001|            17|              NULL|    2020-01-30|

### *Using SQL commands*, limit the output to only 5 rows

In [ ]:
spark.sql("SELECT * FROM patients LIMIT 5").show()

+----------+------+---+-------+--------+-----------+--------------------+-----------+--------------+------------------+--------------+-------------+-------------+--------+
|patient_id|   sex|age|country|province|       city|      infection_case|infected_by|contact_number|symptom_onset_date|confirmed_date|released_date|deceased_date|   state|
+----------+------+---+-------+--------+-----------+--------------------+-----------+--------------+------------------+--------------+-------------+-------------+--------+
|1000000001|  male|50s|  Korea|   Seoul| Gangseo-gu|     overseas inflow|       NULL|            75|        2020-01-22|    2020-01-23|   2020-02-05|         NULL|released|
|1000000002|  male|30s|  Korea|   Seoul|Jungnang-gu|     overseas inflow|       NULL|            31|              NULL|    2020-01-30|   2020-03-02|         NULL|released|
|1000000003|  male|50s|  Korea|   Seoul|  Jongno-gu|contact with patient| 2002000001|            17|              NULL|    2020-01-30|   202

### Select the count of males and females in the dataset

In [ ]:
spark.sql("SELECT sex, count(*) as count FROM patients GROUP BY sex").show()

+------+-----+
|   sex|count|
+------+-----+
|  NULL| 1122|
|female| 2218|
|  male| 1825|
+------+-----+



### How many people did survive, and how many didn't?

In [ ]:
spark.sql("SELECT state, count(*) as count FROM patients GROUP BY state").show()

+--------+-----+
|   state|count|
+--------+-----+
|isolated| 2158|
|released| 2929|
|deceased|   78|
+--------+-----+



### Now, let's perform some preprocessing using SQL:
1. Convert *age* column to double after removing the 's' at the end -- *hint: check SUBSTRING method*
2. Select only the following columns: `['sex', 'age', 'province', 'state']`
3. Store the result of the query in a new dataframe

In [ ]:
df_sql = spark.sql("""
    SELECT sex,
           CAST(SUBSTRING(age, 1, LENGTH(age)-1) AS DOUBLE) as age,
           province,
           state
    FROM patients
""")

In [ ]:
df_sql.show(5)

+------+----+--------+--------+
|   sex| age|province|   state|
+------+----+--------+--------+
|  male|50.0|   Seoul|released|
|  male|30.0|   Seoul|released|
|  male|50.0|   Seoul|released|
|  male|20.0|   Seoul|released|
|female|20.0|   Seoul|released|
+------+----+--------+--------+
only showing top 5 rows


## Machine Learning  
### Create a pipeline model to predict is_dead and evaluate the performance.
- Use **StringIndexer** to transform **string** data type to indices.
- Use **OneHotEncoder** to deal with categorical values.
- Use **Imputer** to fill missing data with mean.

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, Imputer
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql.functions import col

In [ ]:
df_cleaned.printSchema()

root
 |-- age: double (nullable = true)
 |-- province: string (nullable = true)
 |-- is_male: boolean (nullable = false)
 |-- is_dead: boolean (nullable = true)



In [ ]:
# Cast boolean columns to double for ML compatibility
ml_df = df_cleaned.withColumn('is_male', col('is_male').cast('double')).withColumn('is_dead', col('is_dead').cast('double'))

In [ ]:
trainDF, testDF = ml_df.randomSplit([0.8, 0.2], seed=42)
print(f'Train rows: {trainDF.count()}, Test rows: {testDF.count()}')

Train rows: 3293, Test rows: 750


In [ ]:
strIdx = StringIndexer(inputCol='province', outputCol='province_Index', handleInvalid='skip')
ohe = OneHotEncoder(inputCol='province_Index', outputCol='province_OHE')
imputer = Imputer(inputCols=['age'], outputCols=['age_imputed'])
assembler = VectorAssembler(inputCols=['is_male', 'age_imputed', 'province_OHE'], outputCol='features')

In [ ]:
lr = LogisticRegression(featuresCol='features', labelCol='is_dead')
pl = Pipeline(stages=[strIdx, ohe, imputer, assembler, lr])

In [ ]:
pl_model = pl.fit(trainDF)

In [ ]:
train_pred = pl_model.transform(trainDF)
test_pred  = pl_model.transform(testDF)

In [ ]:
auc_eval  = BinaryClassificationEvaluator(labelCol='is_dead',
                                           rawPredictionCol='rawPrediction',
                                           metricName='areaUnderROC')
acc_eval  = MulticlassClassificationEvaluator(labelCol='is_dead',
                                               predictionCol='prediction',
                                               metricName='accuracy')

In [ ]:
print('=== Training Performance ===')
print(f'  AUC  : {auc_eval.evaluate(train_pred):.4f}')
print(f'  Accuracy: {acc_eval.evaluate(train_pred):.4f}')

print('=== Test Performance ===')
print(f'  AUC  : {auc_eval.evaluate(test_pred):.4f}')
print(f'  Accuracy: {acc_eval.evaluate(test_pred):.4f}')

test_pred.select('age', 'province', 'is_male', 'is_dead', 'prediction').show(10)

=== Training Performance ===
  AUC  : 0.9270
  Accuracy: 0.8788
=== Test Performance ===
  AUC  : 0.9170
  Accuracy: 0.8747
+----+----------------+-------+-------+----------+
| age|        province|is_male|is_dead|prediction|
+----+----------------+-------+-------+----------+
|NULL|Gyeongsangbuk-do|    0.0|    0.0|       0.0|
|NULL|Gyeongsangbuk-do|    1.0|    0.0|       0.0|
|NULL|Gyeongsangbuk-do|    1.0|    0.0|       0.0|
|NULL|         Incheon|    0.0|    1.0|       1.0|
|NULL|         Incheon|    0.0|    1.0|       1.0|
|NULL|         Incheon|    0.0|    1.0|       1.0|
|NULL|         Incheon|    0.0|    1.0|       1.0|
|NULL|         Incheon|    0.0|    1.0|       1.0|
|NULL|         Incheon|    0.0|    1.0|       1.0|
|NULL|         Incheon|    0.0|    1.0|       1.0|
+----+----------------+-------+-------+----------+
only showing top 10 rows
